In [1]:
import pandas as pd
import sqlite3

master = pd.read_csv('../data/clean/master.csv')
conn = sqlite3.connect('../data/olist.db')
master.to_sql('master', conn, if_exists='replace', index=False)

pd.read_sql('SELECT COUNT(*) AS n_rows FROM master', conn)

,n_rows
0,109872


In [2]:
query = '''
WITH monthly AS (
  SELECT strftime('%Y-%m', order_purchase_timestamp) AS month,
         COUNT(DISTINCT order_id) AS orders,
         SUM(price) AS revenue
  FROM master
  GROUP BY month
)
SELECT month, orders, ROUND(revenue, 0) AS revenue,
       ROUND(revenue / orders, 2) AS aov,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
             / LAG(revenue) OVER (ORDER BY month), 1) AS mom_pct
FROM monthly
ORDER BY month;
'''
pd.read_sql(query, conn)

,month,orders,revenue,aov,mom_pct
0,2017-01,750,111798.0,149.06,NaN
1,2017-02,1653,234223.0,141.70,109.5
2,2017-03,2546,359199.0,141.08,53.4
3,2017-04,2303,340670.0,147.92,-5.2
4,2017-05,3545,489159.0,137.99,43.6
5,2017-06,3135,421923.0,134.58,-13.7
6,2017-07,3872,481605.0,124.38,14.1
7,2017-08,4193,554700.0,132.29,15.2
8,2017-09,4150,607400.0,146.36,9.5
9,2017-10,4478,648248.0,144.76,6.7


In [10]:
query2 = '''
SELECT category_en,
       ROUND(SUM(CASE WHEN month = '2017-11' THEN price ELSE 0 END), 0) AS nov_rev,
       ROUND(SUM(CASE WHEN month = '2017-12' THEN price ELSE 0 END), 0) AS dec_rev,
       ROUND(SUM(CASE WHEN month = '2017-12' THEN price ELSE 0 END)
           - SUM(CASE WHEN month = '2017-11' THEN price ELSE 0 END), 0) AS change
FROM (
  SELECT category_en, price,
         strftime('%Y-%m', order_purchase_timestamp) AS month
  FROM master
)
GROUP BY category_en
ORDER BY change ASC
LIMIT 10;
'''
pd.read_sql(query2, conn)

,category_en,nov_rev,dec_rev,change
0,bed_bath_table,87958.0,50081.0,-37876.0
1,computers_accessories,69676.0,37429.0,-32248.0
2,furniture_decor,62091.0,31073.0,-31018.0
3,watches_gifts,95292.0,69557.0,-25735.0
4,cool_stuff,55697.0,37516.0,-18181.0
5,health_beauty,78274.0,60689.0,-17586.0
6,garden_tools,44275.0,26796.0,-17479.0
7,telephony,26118.0,12780.0,-13337.0
8,office_furniture,17919.0,6684.0,-11235.0
9,agro_industry_and_commerce,13932.0,5292.0,-8641.0


In [11]:
query3 = '''
WITH first_order AS (
  SELECT customer_unique_id,
         MIN(strftime('%Y-%m', order_purchase_timestamp)) AS first_month
  FROM master
  GROUP BY customer_unique_id
)
SELECT strftime('%Y-%m', m.order_purchase_timestamp) AS month,
       COUNT(DISTINCT CASE WHEN f.first_month = strftime('%Y-%m', m.order_purchase_timestamp)
             THEN m.customer_unique_id END) AS new_customers,
       COUNT(DISTINCT CASE WHEN f.first_month <> strftime('%Y-%m', m.order_purchase_timestamp)
             THEN m.customer_unique_id END) AS returning_customers
FROM master m
JOIN first_order f USING (customer_unique_id)
GROUP BY month
ORDER BY month;
'''
pd.read_sql(query3, conn)

,month,new_customers,returning_customers
0,2017-01,718,0
1,2017-02,1628,2
2,2017-03,2503,5
3,2017-04,2257,17
4,2017-05,3450,28
5,2017-06,3037,39
6,2017-07,3753,49
7,2017-08,4057,57
8,2017-09,4005,78
9,2017-10,4328,89


In [5]:
query3 = '''
WITH first_order AS (
  SELECT customer_unique_id,
         MIN(strftime('%Y-%m', order_purchase_timestamp)) AS first_month
  FROM master
  GROUP BY customer_unique_id
)
SELECT strftime('%Y-%m', m.order_purchase_timestamp) AS month,
       COUNT(DISTINCT CASE WHEN f.first_month = strftime('%Y-%m', m.order_purchase_timestamp)
             THEN m.customer_unique_id END) AS new_customers,
       COUNT(DISTINCT CASE WHEN f.first_month <> strftime('%Y-%m', m.order_purchase_timestamp)
             THEN m.customer_unique_id END) AS returning_customers
FROM master m
JOIN first_order f USING (customer_unique_id)
GROUP BY month
ORDER BY month;
'''
pd.read_sql(query3, conn)

,month,new_customers,returning_customers
0,2017-01,718,0
1,2017-02,1628,2
2,2017-03,2503,5
3,2017-04,2257,17
4,2017-05,3450,28
5,2017-06,3037,39
6,2017-07,3753,49
7,2017-08,4057,57
8,2017-09,4005,78
9,2017-10,4328,89


In [6]:
q5 = '''
SELECT is_late,
       COUNT(*) AS orders,
       ROUND(AVG(review_score), 2) AS avg_review
FROM (SELECT DISTINCT order_id, is_late, review_score FROM master)
GROUP BY is_late;
'''
pd.read_sql(q5, conn)

,is_late,orders,avg_review
0,0,88381,4.29
1,1,7822,2.57


In [7]:
q6 = '''
WITH cat AS (
  SELECT strftime('%Y-%m', order_purchase_timestamp) AS month,
         category_en, SUM(price) AS revenue
  FROM master GROUP BY 1, 2
),
ranked AS (
  SELECT *, RANK() OVER (PARTITION BY month ORDER BY revenue DESC) AS rnk
  FROM cat
)
SELECT month, category_en, ROUND(revenue, 0) AS revenue, rnk
FROM ranked WHERE rnk <= 3
ORDER BY month, rnk;
'''
pd.read_sql(q6, conn)

,month,category_en,revenue,rnk
0,2017-01,furniture_decor,13037.0,1
1,2017-01,health_beauty,12366.0,2
2,2017-01,sports_leisure,9212.0,3
3,2017-02,health_beauty,22060.0,1
4,2017-02,furniture_decor,19926.0,2
5,2017-02,sports_leisure,15630.0,3
6,2017-03,computers_accessories,28079.0,1
7,2017-03,health_beauty,25652.0,2
8,2017-03,sports_leisure,25473.0,3
9,2017-04,bed_bath_table,24093.0,1


In [8]:
q7 = '''
WITH monthly AS (
  SELECT strftime('%Y-%m', order_purchase_timestamp) AS month,
         SUM(price) AS revenue
  FROM master GROUP BY month
)
SELECT month, ROUND(revenue, 0) AS revenue,
       ROUND(SUM(revenue) OVER (ORDER BY month), 0) AS running_total
FROM monthly
ORDER BY month;
'''
pd.read_sql(q7, conn)

,month,revenue,running_total
0,2017-01,111798.0,111798.0
1,2017-02,234223.0,346022.0
2,2017-03,359199.0,705221.0
3,2017-04,340670.0,1045890.0
4,2017-05,489159.0,1535050.0
5,2017-06,421923.0,1956973.0
6,2017-07,481605.0,2438577.0
7,2017-08,554700.0,2993277.0
8,2017-09,607400.0,3600677.0
9,2017-10,648248.0,4248924.0


In [9]:
q8 = '''
WITH cust AS (
  SELECT customer_unique_id, SUM(price) AS spend
  FROM master GROUP BY customer_unique_id
)
SELECT customer_unique_id, ROUND(spend, 0) AS spend,
       ROUND(100.0 * spend / (SELECT SUM(spend) FROM cust), 2) AS share_pct
FROM cust
ORDER BY spend DESC
LIMIT 10;
'''
pd.read_sql(q8, conn)


,customer_unique_id,spend,share_pct
0,0a0a92112bd4c708ca5fde585afaa872,13440.0,0.10
1,da122df9eeddfedc1dc1f5349a1a690c,7388.0,0.06
2,763c8b1c9c68a0229c42c9fc6f662b93,7160.0,0.05
3,dc4802a71eae9be1dd28f5d788ceb526,6735.0,0.05
4,459bef486812aa25204be022145caa62,6729.0,0.05
5,ff4159b92c40ebe40454e3e6a7c35ed6,6499.0,0.05
6,4007669dec559734d6f53e029e360987,5935.0,0.05
7,eebb5dda148d3893cdaf5b5ca3040ccb,4690.0,0.04
8,48e1ac109decbb87765a3eade6854098,4590.0,0.03
9,a229eba70ec1c2abef51f04987deb7a5,4400.0,0.03


In [13]:
query3 = '''
WITH first_order AS (
  SELECT customer_unique_id,
         MIN(strftime('%Y-%m', order_purchase_timestamp)) AS first_month
  FROM master
  GROUP BY customer_unique_id
)
SELECT strftime('%Y-%m', m.order_purchase_timestamp) AS month,
       COUNT(DISTINCT CASE WHEN f.first_month = strftime('%Y-%m', m.order_purchase_timestamp)
             THEN m.customer_unique_id END) AS new_customers,
       COUNT(DISTINCT CASE WHEN f.first_month <> strftime('%Y-%m', m.order_purchase_timestamp)
             THEN m.customer_unique_id END) AS returning_customers
FROM master m
JOIN first_order f USING (customer_unique_id)
GROUP BY month
ORDER BY month;
'''
pd.read_sql(query3, conn)

,month,new_customers,returning_customers
0,2017-01,718,0
1,2017-02,1628,2
2,2017-03,2503,5
3,2017-04,2257,17
4,2017-05,3450,28
5,2017-06,3037,39
6,2017-07,3753,49
7,2017-08,4057,57
8,2017-09,4005,78
9,2017-10,4328,89
